# SignLearn-07 — Error Analysis and Motion Features

This notebook tests whether explicit motion and relative geometry improve signer-independent recognition. It compares two models under the same split, architecture, mirror augmentation, training budget, and seed:

- **Control:** the existing 189 position/detection features.
- **Enhanced:** the same 189 features plus masked hand/pose velocity, acceleration, and 15 relative-geometry measurements (492 total).

WLASL is not loaded or used. It remains reserved for final external evaluation.

In [ ]:
from pathlib import Path
from IPython.display import display
import json
import shutil
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support,
    top_k_accuracy_score
)

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
SEED = 42
BATCH_SIZE = 128
EPOCHS = 32
AUGMENT_PROBABILITY = 0.50
MIRROR_MODE = 'reflect_and_swap_hands'
TARGET_ACCEPTED_ACCURACY = 0.85
MINIMUM_COVERAGE = 0.10

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

KAGGLE_INPUT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/signlearn_07')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Output:', OUTPUT_DIR)

## 1. Load the frozen signer split and deployment metadata

Attach `mvp50_temporal_48_frames.npz` from SignLearn-03 and the latest SignLearn deployment bundle. The notebook refuses a cache whose path contains `wlasl`.

In [ ]:
EXTRACTED_BUNDLE = OUTPUT_DIR / 'input_bundle'
bundle_candidates = []
for zip_path in KAGGLE_INPUT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = archive.namelist()
        has_model = any(name.endswith('signlearn_model.keras') for name in names)
        has_labels = any(name.endswith('label_map.json') for name in names)
        if has_model and has_labels:
            score = 2 if any(name.endswith('camera_orientation_policy.json') for name in names) else 1
            bundle_candidates.append((score, zip_path))
    except zipfile.BadZipFile:
        continue
if bundle_candidates and not EXTRACTED_BUNDLE.exists():
    selected_bundle_zip = sorted(bundle_candidates, key=lambda item: item[0], reverse=True)[0][1]
    with zipfile.ZipFile(selected_bundle_zip) as archive:
        archive.extractall(EXTRACTED_BUNDLE)
    print('Extracted:', selected_bundle_zip)

SEARCH_ROOTS = [KAGGLE_INPUT, EXTRACTED_BUNDLE]

def find_artifact(names, required=True):
    names = [names] if isinstance(names, str) else list(names)
    for name in names:
        for root in SEARCH_ROOTS:
            if root.exists():
                matches = list(root.rglob(name))
                if matches:
                    return matches[0]
    if required:
        raise FileNotFoundError(f'Missing {names}. Attach SignLearn-03 output and the deployment bundle.')
    return None

CACHE_PATH = find_artifact('mvp50_temporal_48_frames.npz')
LABEL_MAP_PATH = find_artifact('label_map.json')
TEMPORAL_CONFIG_PATH = find_artifact('temporal_configuration.json')
BASE_FEATURE_NAMES_PATH = find_artifact('frame_feature_names.json')
PREVIOUS_MANIFEST_PATH = find_artifact('deployment_manifest.json', required=False)
assert 'wlasl' not in str(CACHE_PATH).lower(), 'WLASL must remain external test data.'
print('Cache:', CACHE_PATH)
print('Label map:', LABEL_MAP_PATH)

In [ ]:
cache = np.load(CACHE_PATH, allow_pickle=False)
X = cache['X']
y = cache['y'].astype(np.int32)
splits = cache['splits'].astype(str)
participant_ids = cache['participant_ids']
sequence_ids = cache['sequence_ids']
with LABEL_MAP_PATH.open('r', encoding='utf-8') as stream:
    label_map = json.load(stream)
with BASE_FEATURE_NAMES_PATH.open('r', encoding='utf-8') as stream:
    base_feature_names = json.load(stream)
index_to_sign = {int(index): sign for sign, index in label_map.items()}

train_mask = splits == 'train'
validation_mask = splits == 'validation'
test_mask = splits == 'test'
X_train, y_train = X[train_mask], y[train_mask]
X_validation, y_validation = X[validation_mask], y[validation_mask]
X_test, y_test = X[test_mask], y[test_mask]

participant_sets = {name: set(participant_ids[splits == name].tolist()) for name in ['train', 'validation', 'test']}
assert participant_sets['train'].isdisjoint(participant_sets['validation'])
assert participant_sets['train'].isdisjoint(participant_sets['test'])
assert participant_sets['validation'].isdisjoint(participant_sets['test'])
assert X.shape[1:] == (48, 189)
assert len(base_feature_names) == 189
assert sorted(label_map.values()) == list(range(50))
display(pd.DataFrame({
    'sequences': [train_mask.sum(), validation_mask.sum(), test_mask.sum()],
    'signers': [len(participant_sets[name]) for name in ['train', 'validation', 'test']],
}, index=['train', 'validation', 'test']))
print('Base tensor:', X.shape, X.dtype)
print('Test labels will not be used until model selection is frozen.')

## 2. Define mirror augmentation and enhanced features

Velocity and acceleration are calculated for both hands and the selected pose points. Motion is masked when the relevant landmarks are missing in adjacent frames, preventing missing-to-zero transitions from becoming false movement. Relative distances are scale invariant because the base coordinates are shoulder-normalized.

In [ ]:
BASE_FEATURE_COUNT = 189
MOTION_COORDINATE_COUNT = 144  # both hands (126) + selected pose xy (18)
GEOMETRY_FEATURE_NAMES = [
    'wrist_to_wrist_distance',
    'left_wrist_to_mouth_distance', 'right_wrist_to_mouth_distance',
    'left_wrist_to_nose_distance', 'right_wrist_to_nose_distance',
    'left_wrist_to_shoulder_distance', 'right_wrist_to_shoulder_distance',
    'left_hand_spread', 'right_hand_spread',
    'left_thumb_index_distance', 'right_thumb_index_distance',
    'left_hand_to_pose_wrist_distance', 'right_hand_to_pose_wrist_distance',
    'left_fingertip_to_mouth_min_distance', 'right_fingertip_to_mouth_min_distance',
]
ENHANCED_FEATURE_COUNT = BASE_FEATURE_COUNT + 2 * MOTION_COORDINATE_COUNT + len(GEOMETRY_FEATURE_NAMES)
assert ENHANCED_FEATURE_COUNT == 492

reflection_sign = np.ones(BASE_FEATURE_COUNT, dtype=np.float32)
reflection_sign[0:63:3] = -1.0
reflection_sign[63:126:3] = -1.0
reflection_sign[126:144:2] = -1.0
reflection_sign[144:184:2] = -1.0
mirror_permutation = np.arange(BASE_FEATURE_COUNT, dtype=np.int32)
mirror_permutation[0:63] = np.arange(63, 126)
mirror_permutation[63:126] = np.arange(0, 63)
mirror_permutation[184], mirror_permutation[185] = 185, 184

TF_REFLECTION_SIGN = tf.constant(reflection_sign, tf.float32)
TF_MIRROR_PERMUTATION = tf.constant(mirror_permutation, tf.int32)

def mirror_base_features(features):
    return tf.gather(features, TF_MIRROR_PERMUTATION, axis=-1) * TF_REFLECTION_SIGN

def vector_distance(first, second):
    return tf.sqrt(tf.reduce_sum(tf.square(first - second), axis=-1) + 1e-8)

def build_enhanced_features(features):
    features = tf.cast(features, tf.float32)
    motion_coordinates = features[:, :MOTION_COORDINATE_COUNT]
    detection = features[:, 184:189]

    previous_coordinates = tf.concat([motion_coordinates[:1], motion_coordinates[:-1]], axis=0)
    velocity = motion_coordinates - previous_coordinates
    previous_detection = tf.concat([detection[:1], detection[:-1]], axis=0)
    velocity_masks = tf.concat([
        tf.repeat((detection[:, 0:1] > 0.5) & (previous_detection[:, 0:1] > 0.5), 63, axis=1),
        tf.repeat((detection[:, 1:2] > 0.5) & (previous_detection[:, 1:2] > 0.5), 63, axis=1),
        tf.repeat((detection[:, 2:3] > 0.5) & (previous_detection[:, 2:3] > 0.5), 18, axis=1),
    ], axis=1)
    velocity = tf.where(velocity_masks, velocity, 0.0)

    previous_velocity = tf.concat([velocity[:1], velocity[:-1]], axis=0)
    acceleration = velocity - previous_velocity
    previous_velocity_mask = tf.concat([velocity_masks[:1], velocity_masks[:-1]], axis=0)
    acceleration = tf.where(velocity_masks & previous_velocity_mask, acceleration, 0.0)
    velocity = tf.clip_by_value(velocity, -5.0, 5.0)
    acceleration = tf.clip_by_value(acceleration, -5.0, 5.0)

    left_hand = tf.reshape(features[:, 0:63], [-1, 21, 3])
    right_hand = tf.reshape(features[:, 63:126], [-1, 21, 3])
    pose = tf.reshape(features[:, 126:144], [-1, 9, 2])
    lips = tf.reshape(features[:, 144:184], [-1, 20, 2])
    left_wrist, right_wrist = left_hand[:, 0, :2], right_hand[:, 0, :2]
    nose, left_shoulder, right_shoulder = pose[:, 0], pose[:, 1], pose[:, 2]
    left_pose_wrist, right_pose_wrist = pose[:, 5], pose[:, 6]
    mouth = tf.reduce_mean(lips, axis=1)
    fingertip_indices = tf.constant([4, 8, 12, 16, 20], tf.int32)
    left_tips = tf.gather(left_hand[:, :, :2], fingertip_indices, axis=1)
    right_tips = tf.gather(right_hand[:, :, :2], fingertip_indices, axis=1)
    left_spread = tf.reduce_mean(vector_distance(left_tips, left_wrist[:, None, :]), axis=1)
    right_spread = tf.reduce_mean(vector_distance(right_tips, right_wrist[:, None, :]), axis=1)
    left_tip_to_mouth = tf.reduce_min(vector_distance(left_tips, mouth[:, None, :]), axis=1)
    right_tip_to_mouth = tf.reduce_min(vector_distance(right_tips, mouth[:, None, :]), axis=1)

    geometry = tf.stack([
        vector_distance(left_wrist, right_wrist),
        vector_distance(left_wrist, mouth), vector_distance(right_wrist, mouth),
        vector_distance(left_wrist, nose), vector_distance(right_wrist, nose),
        vector_distance(left_wrist, left_shoulder), vector_distance(right_wrist, right_shoulder),
        left_spread, right_spread,
        vector_distance(left_hand[:, 4, :2], left_hand[:, 8, :2]),
        vector_distance(right_hand[:, 4, :2], right_hand[:, 8, :2]),
        vector_distance(left_wrist, left_pose_wrist), vector_distance(right_wrist, right_pose_wrist),
        left_tip_to_mouth, right_tip_to_mouth,
    ], axis=1)

    left_valid = detection[:, 0] > 0.5
    right_valid = detection[:, 1] > 0.5
    pose_valid = detection[:, 2] > 0.5
    lip_valid = detection[:, 3] > 0.5
    geometry_masks = tf.stack([
        left_valid & right_valid,
        left_valid & lip_valid, right_valid & lip_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid, right_valid, left_valid, right_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid & lip_valid, right_valid & lip_valid,
    ], axis=1)
    geometry = tf.where(geometry_masks, tf.clip_by_value(geometry, 0.0, 10.0), 0.0)
    enhanced = tf.concat([features, velocity, acceleration, geometry], axis=1)
    return tf.ensure_shape(enhanced, [48, ENHANCED_FEATURE_COUNT])

enhanced_feature_names = (
    base_feature_names
    + [f'velocity_{name}' for name in base_feature_names[:MOTION_COORDINATE_COUNT]]
    + [f'acceleration_{name}' for name in base_feature_names[:MOTION_COORDINATE_COUNT]]
    + GEOMETRY_FEATURE_NAMES
)
assert len(enhanced_feature_names) == ENHANCED_FEATURE_COUNT

In [ ]:
sample = tf.cast(X_validation[0], tf.float32)
enhanced_sample = build_enhanced_features(sample)
mirrored_twice = mirror_base_features(mirror_base_features(sample))
assert enhanced_sample.shape == (48, 492)
assert bool(tf.reduce_all(tf.math.is_finite(enhanced_sample)))
np.testing.assert_allclose(mirrored_twice.numpy(), sample.numpy(), atol=1e-6)
assert np.allclose(enhanced_sample.numpy()[0, 189:189 + 288], 0.0)
print('Enhanced feature shape:', enhanced_sample.shape)
print('Finite feature contract and mirror involution: OK')

## 3. Build identical control and enhanced datasets

Both training pipelines first apply the same random mirror transformation. Only the enhanced pipeline then derives motion and geometry. Validation and test transformations are deterministic.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def randomly_mirror(features, label):
    features = tf.cast(features, tf.float32)
    use_mirror = tf.random.uniform([], seed=SEED) < AUGMENT_PROBABILITY
    return tf.cond(use_mirror, lambda: mirror_base_features(features), lambda: features), label

def add_enhanced_features(features, label):
    return build_enhanced_features(features), label

def training_dataset(enhanced=False):
    dataset = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
        .map(randomly_mirror, num_parallel_calls=AUTOTUNE, deterministic=True)
    )
    if enhanced:
        dataset = dataset.map(add_enhanced_features, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def evaluation_dataset(features, labels, enhanced=False, mirrored=False):
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    if mirrored:
        dataset = dataset.map(
            lambda item, label: (mirror_base_features(tf.cast(item, tf.float32)), label),
            num_parallel_calls=AUTOTUNE, deterministic=True
        )
    if enhanced:
        dataset = dataset.map(add_enhanced_features, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

control_train = training_dataset(enhanced=False)
enhanced_train = training_dataset(enhanced=True)
control_validation = evaluation_dataset(X_validation, y_validation, enhanced=False)
enhanced_validation = evaluation_dataset(X_validation, y_validation, enhanced=True)
print('Control input:', next(iter(control_train))[0].shape)
print('Enhanced input:', next(iter(enhanced_train))[0].shape)

## 4. Train the controlled ablation

The architectures differ only in input width. Model selection uses validation macro F1, not test or WLASL performance.

In [ ]:
def build_temporal_model(feature_count, model_name):
    inputs = tf.keras.Input((48, feature_count), dtype=tf.float32, name='landmark_sequence')
    x = tf.keras.layers.GaussianNoise(0.015, name='landmark_noise')(inputs)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal', name='frame_projection')(x)
    x = tf.keras.layers.LayerNormalization(name='frame_normalization')(x)
    x = tf.keras.layers.SpatialDropout1D(0.15)(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(96, return_sequences=True, dropout=0.20), name='bilstm_1'
    )(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.20), name='bilstm_2'
    )(x)
    attention = tf.keras.layers.MultiHeadAttention(
        num_heads=4, key_dim=32, dropout=0.15, name='temporal_self_attention'
    )(x, x)
    x = tf.keras.layers.LayerNormalization(name='attention_residual')(x + attention)
    average_pool = tf.keras.layers.GlobalAveragePooling1D()(x)
    maximum_pool = tf.keras.layers.GlobalMaxPooling1D()(x)
    x = tf.keras.layers.Concatenate()([average_pool, maximum_pool])
    x = tf.keras.layers.Dense(
        192, activation='relu', kernel_initializer='he_normal',
        kernel_regularizer=tf.keras.regularizers.l2(2e-5)
    )(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(50, activation='softmax', name='sign_probabilities')(x)
    model = tf.keras.Model(inputs, outputs, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=7e-4, clipnorm=1.0),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5_accuracy'),
        ],
    )
    return model

def train_candidate(name, feature_count, train_data, validation_data):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = build_temporal_model(feature_count, f'signlearn_{name}')
    path = OUTPUT_DIR / f'best_{name}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_loss', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]
    started = time.time()
    history = model.fit(
        train_data, validation_data=validation_data, epochs=EPOCHS, callbacks=callbacks, verbose=2
    )
    history_frame = pd.DataFrame(history.history)
    history_frame['candidate'] = name
    history_frame.to_csv(OUTPUT_DIR / f'{name}_history.csv', index=False)
    print(f'{name} training minutes: {(time.time() - started) / 60:.1f}')
    return path, history_frame

control_path, control_history = train_candidate(
    'control_189', 189, control_train, control_validation
)
enhanced_path, enhanced_history = train_candidate(
    'motion_geometry_492', 492, enhanced_train, enhanced_validation
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for history_frame, label in [(control_history, 'control'), (enhanced_history, 'enhanced')]:
    axes[0].plot(history_frame['val_loss'], label=label)
    axes[1].plot(history_frame['val_accuracy'], label=label)
axes[0].set(title='Validation loss', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Validation accuracy', xlabel='Epoch', ylabel='Accuracy')
for axis in axes:
    axis.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'controlled_ablation_training.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Validation selection and per-class error analysis

The winner is frozen using original-view validation macro F1. Mirrored-view performance is reported as a robustness diagnostic.

In [ ]:
def predict_dataset(model, dataset):
    return model.predict(dataset.map(lambda features, labels: features), verbose=0)

def probability_metrics(true_labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    return {
        'accuracy': float(accuracy_score(true_labels, predictions)),
        'macro_f1': float(f1_score(true_labels, predictions, average='macro')),
        'top5_accuracy': float(top_k_accuracy_score(
            true_labels, probabilities, k=5, labels=np.arange(50)
        )),
    }

control_model = tf.keras.models.load_model(control_path, compile=False)
enhanced_model = tf.keras.models.load_model(enhanced_path, compile=False)
validation_sets = {
    'control_189': (control_model, control_validation, evaluation_dataset(
        X_validation, y_validation, enhanced=False, mirrored=True
    )),
    'motion_geometry_492': (enhanced_model, enhanced_validation, evaluation_dataset(
        X_validation, y_validation, enhanced=True, mirrored=True
    )),
}
validation_probabilities = {}
validation_rows = []
for name, (model, original_dataset, mirrored_dataset) in validation_sets.items():
    original = predict_dataset(model, original_dataset)
    mirrored = predict_dataset(model, mirrored_dataset)
    validation_probabilities[name] = original
    original_predictions = original.argmax(axis=1)
    mirrored_predictions = mirrored.argmax(axis=1)
    validation_rows.append({
        'candidate': name, 'view': 'original', **probability_metrics(y_validation, original),
        'agreement_with_original': 1.0,
    })
    validation_rows.append({
        'candidate': name, 'view': 'mirrored', **probability_metrics(y_validation, mirrored),
        'agreement_with_original': float((mirrored_predictions == original_predictions).mean()),
    })

validation_comparison = pd.DataFrame(validation_rows)
display(validation_comparison.style.format({
    'accuracy': '{:.4f}', 'macro_f1': '{:.4f}', 'top5_accuracy': '{:.4f}',
    'agreement_with_original': '{:.4f}'
}))
validation_comparison.to_csv(OUTPUT_DIR / 'validation_feature_ablation.csv', index=False)
original_rows = validation_comparison[validation_comparison['view'] == 'original']
WINNER = original_rows.sort_values('macro_f1', ascending=False).iloc[0]['candidate']
print('Frozen winner selected on validation macro F1:', WINNER)

In [ ]:
per_class_frames = []
for name, probabilities in validation_probabilities.items():
    predictions = probabilities.argmax(axis=1)
    precision, recall, f1_values, support = precision_recall_fscore_support(
        y_validation, predictions, labels=np.arange(50), zero_division=0
    )
    per_class_frames.append(pd.DataFrame({
        'candidate': name, 'label': np.arange(50),
        'sign': [index_to_sign[index] for index in range(50)],
        'precision': precision, 'recall': recall, 'f1': f1_values, 'support': support,
    }))
per_class = pd.concat(per_class_frames, ignore_index=True)
per_class.to_csv(OUTPUT_DIR / 'validation_per_class_metrics.csv', index=False)
pivot = per_class.pivot(index='sign', columns='candidate', values='f1')
pivot['delta_enhanced_minus_control'] = pivot['motion_geometry_492'] - pivot['control_189']
display(pivot.sort_values('delta_enhanced_minus_control', ascending=False).head(10).style.format('{:.3f}'))
display(pivot.sort_values('delta_enhanced_minus_control').head(10).style.format('{:.3f}'))
pivot.to_csv(OUTPUT_DIR / 'validation_per_class_f1_delta.csv')

## 6. Select the winner's confidence threshold on validation

The lowest threshold reaching the target accepted validation accuracy and minimum coverage is frozen. This policy is selected before test evaluation.

In [ ]:
winner_validation_probabilities = validation_probabilities[WINNER]

def selective_curve(probabilities, true_labels, thresholds):
    predictions = probabilities.argmax(axis=1)
    confidences = probabilities.max(axis=1)
    rows = []
    for threshold in thresholds:
        accepted = confidences >= threshold
        rows.append({
            'threshold': float(threshold),
            'coverage': float(accepted.mean()),
            'accepted_accuracy': (
                float((predictions[accepted] == true_labels[accepted]).mean())
                if accepted.any() else np.nan
            ),
            'accepted_count': int(accepted.sum()),
        })
    return pd.DataFrame(rows)

validation_selective = selective_curve(
    winner_validation_probabilities, y_validation, np.arange(0.0, 1.0, 0.01)
)
eligible = validation_selective[
    (validation_selective['accepted_accuracy'] >= TARGET_ACCEPTED_ACCURACY)
    & (validation_selective['coverage'] >= MINIMUM_COVERAGE)
]
if eligible.empty:
    chosen_threshold_row = validation_selective.sort_values(
        ['accepted_accuracy', 'coverage'], ascending=False
    ).iloc[0]
    print('Warning: target accepted accuracy not reached; using best validation row.')
else:
    chosen_threshold_row = eligible.sort_values('threshold').iloc[0]
CONFIDENCE_THRESHOLD = float(chosen_threshold_row['threshold'])
validation_selective.to_csv(OUTPUT_DIR / 'winner_validation_selective_curve.csv', index=False)
display(chosen_threshold_row.to_frame('frozen_validation_policy'))
print('Frozen winner:', WINNER)
print('Frozen threshold:', CONFIDENCE_THRESHOLD)

## 7. Evaluate both frozen candidates once on Google test signers

Model selection and the threshold are already frozen. Do not revise them based on the following results. WLASL remains untouched.

In [ ]:
test_sets = {
    'control_189': (control_model, evaluation_dataset(X_test, y_test, enhanced=False)),
    'motion_geometry_492': (enhanced_model, evaluation_dataset(X_test, y_test, enhanced=True)),
}
test_probabilities = {}
test_rows = []
for name, (model, dataset) in test_sets.items():
    probabilities = predict_dataset(model, dataset)
    test_probabilities[name] = probabilities
    test_rows.append({'candidate': name, **probability_metrics(y_test, probabilities)})
test_comparison = pd.DataFrame(test_rows).set_index('candidate')
display(test_comparison.style.format('{:.4f}'))
test_comparison.to_csv(OUTPUT_DIR / 'frozen_google_test_feature_ablation.csv')

winner_test_probabilities = test_probabilities[WINNER]
winner_test_predictions = winner_test_probabilities.argmax(axis=1)
winner_test_confidences = winner_test_probabilities.max(axis=1)
accepted = winner_test_confidences >= CONFIDENCE_THRESHOLD
winner_test_metrics = {
    **probability_metrics(y_test, winner_test_probabilities),
    'coverage_at_frozen_threshold': float(accepted.mean()),
    'accepted_accuracy_at_frozen_threshold': (
        float((winner_test_predictions[accepted] == y_test[accepted]).mean()) if accepted.any() else np.nan
    ),
    'accepted_count': int(accepted.sum()),
}
display(pd.Series(winner_test_metrics, name='frozen_winner_test').to_frame().style.format('{:.4f}'))

test_predictions_frame = pd.DataFrame({
    'sequence_id': sequence_ids[test_mask],
    'participant_id': participant_ids[test_mask],
    'true_sign': [index_to_sign[int(value)] for value in y_test],
    'predicted_sign': [index_to_sign[int(value)] for value in winner_test_predictions],
    'confidence': winner_test_confidences, 'accepted': accepted,
})
test_predictions_frame.to_csv(OUTPUT_DIR / 'winner_frozen_test_predictions.csv', index=False)

In [ ]:
winner_confusion = confusion_matrix(
    y_test, winner_test_predictions, labels=np.arange(50), normalize='true'
)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(winner_confusion, cmap='mako', vmin=0, vmax=1, xticklabels=False, yticklabels=False, ax=ax)
ax.set(title=f'Frozen Google test confusion: {WINNER}', xlabel='Predicted', ylabel='True')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'winner_test_confusion.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Export the validation-selected model

If the enhanced model wins, the application preprocessing must generate the same 492 features before the model can be deployed. The exported feature configuration makes that contract explicit.

In [ ]:
winner_model = control_model if WINNER == 'control_189' else enhanced_model
winner_feature_names = base_feature_names if WINNER == 'control_189' else enhanced_feature_names
winner_feature_count = 189 if WINNER == 'control_189' else 492
EXPORT_DIR = OUTPUT_DIR / 'signlearn_motion_feature_bundle'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
winner_model.save(EXPORT_DIR / 'signlearn_model.keras')

feature_configuration = {
    'feature_version': 'signlearn_07_motion_geometry_v1',
    'base_feature_count': 189,
    'motion_coordinate_count': MOTION_COORDINATE_COUNT,
    'velocity_feature_count': MOTION_COORDINATE_COUNT,
    'acceleration_feature_count': MOTION_COORDINATE_COUNT,
    'geometry_feature_count': len(GEOMETRY_FEATURE_NAMES),
    'selected_feature_count': winner_feature_count,
    'selected_candidate': WINNER,
    'motion_masking': 'adjacent-frame group detection ratio > 0.5',
    'motion_clip': [-5.0, 5.0],
    'geometry_clip': [0.0, 10.0],
    'mirror_mode': MIRROR_MODE,
}
with (EXPORT_DIR / 'feature_configuration.json').open('w', encoding='utf-8') as stream:
    json.dump(feature_configuration, stream, indent=2)
with (EXPORT_DIR / 'frame_feature_names.json').open('w', encoding='utf-8') as stream:
    json.dump(winner_feature_names, stream, indent=2)
with (EXPORT_DIR / 'label_map.json').open('w', encoding='utf-8') as stream:
    json.dump(label_map, stream, indent=2)
shutil.copy2(TEMPORAL_CONFIG_PATH, EXPORT_DIR / 'temporal_configuration.json')

manifest = {
    'project': 'SignLearn ASL Practice Assistant — motion feature ablation',
    'inference_backend': 'tensorflow_keras',
    'input_shape': [1, 48, winner_feature_count],
    'input_dtype': 'float32', 'class_count': 50,
    'selected_candidate': WINNER,
    'selection_metric': 'validation macro F1',
    'mirror_augmentation_mode': MIRROR_MODE,
    'mirror_augmentation_probability': AUGMENT_PROBABILITY,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'validation_selected_policy': {
        key: (int(value) if key == 'accepted_count' else float(value))
        for key, value in chosen_threshold_row.to_dict().items()
    },
    'validation_comparison': validation_comparison.to_dict(orient='records'),
    'frozen_google_test_comparison': test_comparison.reset_index().to_dict(orient='records'),
    'frozen_winner_test_metrics': winner_test_metrics,
    'external_test_status': 'WLASL not loaded or used',
}
with (EXPORT_DIR / 'deployment_manifest.json').open('w', encoding='utf-8') as stream:
    json.dump(manifest, stream, indent=2)

archive_path = shutil.make_archive(
    str(OUTPUT_DIR / 'signlearn_motion_feature_bundle'), 'zip', root_dir=EXPORT_DIR
)
print('Selected candidate:', WINNER)
print('Exported:', archive_path)
for path in sorted(EXPORT_DIR.iterdir()):
    print(f'  {path.name}: {path.stat().st_size:,} bytes')

## Decision rules

1. Keep the enhanced model only if it wins on validation macro F1.
2. Report both control and enhanced frozen test results, including regressions.
3. Do not change the feature design after viewing test results.
4. Do not use WLASL to select features, architecture, thresholds, or camera orientation.
5. If enhanced features do not help, the next priority is a consented webcam-development set and domain adaptation—not a larger network.